In [16]:
# auto-reload changes to imported modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import compile_tables as ct
import compile_file as cf
import parser_dat as p
import pandas as pd
from missing_logk import MissingSolutionSpecies

In [18]:
db_list = p.phreeqc_database_list('../databases/test_data')
print(db_list)
mst = ct.compile_master_solution_table(db_list)
sp = ct.compile_solution_species_table(db_list)
phases = ct.compile_phase_table(db_list)

['../databases/test_data/pitzer.dat', '../databases/test_data/minteq.v4.dat']


In [19]:
mst.head()

,element,species,alk,gfw_formula,element_gfw,source
0,Alkalinity,CO3-2,1,Ca0.5(CO3)0.5,50.05,#pitzer.dat
1,B,B(OH)3,0,B,10.81,#pitzer.dat
2,Ba,Ba+2,0,Ba,137.33,#pitzer.dat
3,Br,Br-,0,Br,79.904,#pitzer.dat
4,C,CO3-2,2,HCO3,12.0111,#pitzer.dat


In [20]:
def custom_func(x):
    return sp[(sp['log_k'] == 0) & (sp['equation'].str.contains(x, regex=False))].empty
custom_func('asdf')

True

In [21]:
# mst['bool'] = mst['species'].apply(custom_func)

In [22]:
# mst[(mst['bool']) & mst['element_gfw'].notnull()]

In [23]:
sp.head()

,equation,log_k,delta_h,gamma,d_w,v_m,no_check,source
0,CO3-2 + H+ = HCO3-,10.3393,"(-3.561, kcal)",None,"(1.18e-09, 0.0, 1.43, 1e-10)","(8.54, 0.0, -11.7, 0.0, 1.6, 0.0, 0.0, 116.0, ...",None,pitzer.dat
1,CO3-2 + 2 H+ = CO2 + H2O,16.6767,"(-5.738, kcal)",None,"(1.92e-09,)","(7.29, 0.92, 2.07, -1.23, -1.6)",None,pitzer.dat
2,SO4-2 + H+ = HSO4-,1.9790,"(4.91, kcal)",None,"(1.33e-09,)","(8.2, 9.259, 2.1108, -3.1618, 1.1748, 0.0, -0....",None,pitzer.dat
3,H2Sg = HSg- + H+,-6.9940,"(5.30, kcal)",None,"(1.73e-09,)","(5.0119, 4.9799, 3.4765, -2.9849, 1.441)",None,pitzer.dat
4,B(OH)3 + H2O = B(OH)4- + H+,-9.2390,"(0, kcal)",None,None,None,None,pitzer.dat


In [24]:
sp[(sp['log_k'] == 0) & (sp['equation'].str.contains('H2O'))]

,equation,log_k,delta_h,gamma,d_w,v_m,no_check,source
14,H2O = H2O,0.0,None,None,None,None,None,minteq.v4.dat


In [25]:
sp[sp['equation'].str.contains('H2Sg')]

,equation,log_k,delta_h,gamma,d_w,v_m,no_check,source
3,H2Sg = HSg- + H+,-6.994,"(5.30, kcal)",None,"(1.73e-09,)","(5.0119, 4.9799, 3.4765, -2.9849, 1.441)",None,pitzer.dat


In [26]:
phases.head()

,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
0,Akermanite,Ca2MgSi2O7 + 6 H+ = Mg+2 + 2 Ca+2 + 2 H4SiO4 -...,45.23,"(-289, kJ/mol)",None,"(92.6,)",None,None,None,pitzer.dat
1,Anhydrite,CaSO4 = Ca+2 + SO4-2,-4.362,None,"(5.009, -2.21e-2, -796.4)","(46.1,)",None,None,None,pitzer.dat
2,Anthophyllite,Mg7Si8O22(OH)2 + 14 H+ = 7 Mg+2 - 8 H2O + 8 H4...,66.8,"(-483, kJ/mol)",None,"(269.0,)",None,None,None,pitzer.dat
3,Antigorite,Mg48Si34O85(OH)62 + 96 H+ = 34 H4SiO4 + 48 Mg+...,477.19,"(-3364, kJ/mol)",None,"(1745.0,)",None,None,None,pitzer.dat
4,Aragonite,CaCO3 = CO3-2 + Ca+2,-8.336,"(-2.589, kcal)","(-171.8607, -.077993, 2903.293, 71.595)","(34.04,)",None,None,None,pitzer.dat


In [27]:
phases[phases['phase_name'].str.contains('MASTER')]

,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source


In [28]:
mss = MissingSolutionSpecies(mst, sp)

MissingSolutionSpecies: Not all species were found in the database
MissingSolutionSpecies: All species were found in the database


In [15]:
MissingSolutionSpecies(mss.mst, mss.sp)

MissingSolutionSpecies: All species were found in the database


In [37]:
work[work['equation'].str.contains('H2Sg')]

,equation,log_k,delta_h,gamma,d_w,v_m,no_check,source
3,H2Sg = HSg- + H+,-6.994,"(5.30, kcal)",None,"(1.73e-09,)","(5.0119, 4.9799, 3.4765, -2.9849, 1.441)",None,pitzer.dat
1348,H2Sg = H2Sg,0.0,None,None,None,None,None,was_missing


In [39]:
w = mss.mst
w['bool']

0      False
1       True
2      False
3      False
4      False
       ...  
117    False
118    False
119    False
120    False
121    False
Name: bool, Length: 129, dtype: bool

In [43]:
~mss.mst['species'].apply(mss.custom_apply).all()

True